# BP7 Gate 5 — Decision Layer & Reporting
**Customer360 Navigator Enterprise Suite — Customer360 Navigator Decision Engine**

## Why this gate is different from every other BP's own Gate 5 in this suite
Master Plan Section 8's generic Gate 5 row ("Decision / GenAI Layer & Reporting") bundles a
GenAI output, a decision-engine score, and reporting. For BP1/BP2/BP3/BP4, the "Decision" part
(a priority/intervention-risk score **combining multiple BPs' outputs**) was explicitly
**NOT_APPLICABLE at that BP's own level** and named, in each of those BPs' own Gate 5 notebooks,
as BP7's job instead (Master Plan Section 5.1/7: *"Customer Navigator Decision Engine —
transparent, auditable decision rules combining BP1-BP5 outputs into a priority score and
intervention flag"*). This is that gate. BP7 Gate 5 is where the cross-BP decision every upstream
BP deferred is finally, actually computed and written — for real, for every one of the real
1,048,575 CFPB rows — using Gate 3's already-selected champion rule scheme
(`correlation_aware_plus_lr_diagnostic`, weights `bp2=0.222714, bp3=0.170774, bp4=0.606512`,
`intervention_threshold=0.5`), now independently re-verified bit-exact by Gate 4 (including Gate
4's own real resolution of the ECOA/Reg B disparate-impact check Gate 3 honestly deferred).

- **"GenAI"** → **Not Applicable**, the identical standing scope decision confirmed at every
  upstream BP's own Gate 5: `recommended_action` is Gate 1's own named deterministic,
  reason-code-keyed lookup — *"never GenAI ... UDAAP Section 9 does not map BP7"* — never a
  generated string. No GenAI API is called anywhere in this notebook.
- **"Decision"** → genuinely applies, and is this gate's real substance: full-population
  `priority_score` / `intervention_flag` / `recommended_action` / `reason_codes`, computed for
  every real complaint.
- **"Reporting"** → also genuinely applies: a real, live-computed `recommended_action` breakdown,
  a BP4-tier × `intervention_flag` cross-tab, the real per-field upstream-coverage report (Gate
  2's own `coverage_report()`, reused unmodified), and a fresh, full-population re-derivation of
  the ECOA/Reg B disparate-impact audit, cross-checked against Gate 4's own recorded finding.

## A real precision issue this gate had to resolve first (never guessed past)
`configs/bp7_customer_navigator_decision_engine.yaml`'s own recorded champion weights
(`champion_weight_bp2: 0.222714`, etc.) are **rounded to 6 decimal places** — `benchmark_candidate()`'s
own `round(..., 6)`, written for human-readable config storage. Gate 3 and Gate 4 themselves always
*scored* using the full-precision, unrounded normalized weights, never the rounded config value.
Reading the rounded config value back in here and scoring the real full population with it would
risk a small (~1e-6-scale) numeric drift from what Gate 3/4 actually validated bit-exact — a real,
avoidable precision bug. This gate avoids it by construction: it never reads the champion's
*weights* from config for scoring — only the champion's **name** (`champion_rule_scheme`) — and
independently **re-derives** the full-precision normalized weights for that one, already-locked-in
candidate via the identical real function chain Gate 3/4 both used
(`compute_bp2_bp3_correlation` → `compute_bp4_join_coverage` → `compute_candidate_raw_weights` →
`normalize_candidate_weights`). It deliberately does **not** re-run `select_champion()` itself —
that already happened twice (Gate 3's own benchmark, Gate 4's own independent reproduction); doing
it a third time here would risk silently landing on a *different* champion if any upstream artifact
had since drifted, which is Gate 3/4's own job to catch, not this gate's. Section 6 below
cross-checks the freshly re-derived weights against the recorded (rounded) config values, with the
same tolerances Gate 4 already established for this identical rounding gap (`1e-6` for the weights
themselves, `1e-4` for `cramers_v`/`bp4_join_coverage`) — and only then uses the fresh,
full-precision weights for the real scoring pass in Section 7.

## What this notebook writes (the real, final BP7 deliverable)
One row **per real complaint** (1,048,575 rows, not a sample and not a held-out test split, unlike
BP1/BP2/BP3's own Gate 5 decision records) carrying `priority_score`, `intervention_flag`,
`recommended_action`, `reason_codes` (Gate 1's own four named output fields), the exact per-row
`contribution_bp2`/`contribution_bp3`/`contribution_bp4` decomposition, every real upstream field
that fed or gave context to that score, and — audit-only, joined strictly after scoring, never a
scoring input — BP3's own real `tags_group` (the identical real precedent Gate 4 already
established and this gate reuses unmodified). Three real reporting rollups accompany it:
`recommended_action` breakdown, BP4-tier × `intervention_flag` cross-tab, and a fresh disparate-
impact breakdown by `tags_group`.

## Standing rules this notebook follows
- **Execution boundary**: Claude wrote this notebook; you run it. Every real number below — every
  re-derived weight, every row of `priority_score`/`intervention_flag`, every reporting rollup —
  is a real measurement from your own machine's run against your own real Gate 2/3/4 artifacts,
  never simulated.
- **Zero-fabrication**: no financial-impact, illustrative, or assumption-based content anywhere;
  no GenAI call; no BP1-6 source file is modified — this gate only reads BP3's own already-real-run
  -confirmed, already-governance-reviewed Gold layer for the audit-only `tags_group` join.
- **WARP**: `configure_performance()` first, before any heavy import; the RAM-ceiling assertion
  Gate 4 used is reused here too (this gate processes the same real 1,048,575-row full population).
- **HYPER**: every new Gate 5 function lives in `src/features/bp7_decision_engine_features.py`
  (extended, not duplicated) and reuses `attach_normalized_signal_columns()` /
  `score_priority_rule()` / `benchmark_candidate()` / `compute_priority_score_contribution_
  decomposition()` / `load_bp3_gold_tags_group()` / `attach_tags_group_for_audit()` /
  `compute_disparate_impact_audit()` / `coverage_report()` from this same module's own Gate
  2/3/4 sections **unmodified**; reuses `src/utils/bp1_config_sync.py` (eighth BP-gate reuse).
- **Idempotent**: re-running this notebook overwrites this gate's own config block and artifacts
  in place; every other gate's block is preserved verbatim regardless of position.
- **`status` is deliberately NOT touched** — this project's own established convention for BP7
  specifically (BP7 Gate 2, Gate 3, and Gate 4 all left `status: "gate1_confirmed"` untouched) is
  that `status` is updated only at a BP's own final gate. This gate follows that same convention.
- **PROJECT_STRUCTURE_LOCKED.md rule #3**: same project-root resolver as every other notebook.
- **No test coverage added here** — matching every other BP's own precedent, BP7's own first
  tests are deferred to its own Gate 6.

## Outputs (idempotent overwrite-in-place)
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate5_full_population_decision_records.csv`
  — one real row per complaint (1,048,575 rows), Gate 1's own four named output fields plus the
  exact contribution decomposition, upstream context fields, and audit-only `tags_group`.
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate5_recommended_action_breakdown.csv`
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate5_bp4_tier_intervention_crosstab.csv`
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate5_disparate_impact_breakdown.csv`
  — freshly re-derived on this gate's own scored full population, cross-checked against Gate 4.
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate5_decision_layer_summary.json`
- `configs/bp7_customer_navigator_decision_engine.yaml` — Gate 5 marker block appended/overwritten
  (`status` untouched).

## Prerequisites
BP7 Gate 4 must have been real-run at least once — this notebook checks the Gate 4 config block
and `gate4_statistical_validation_explainability_summary.json` live and raises a clear error if
either is missing, and specifically requires `gate4_champion_reproduced_bit_exact: True` and
`gate4_leakage_reconfirmed_clean: True` to be recorded before it will compute anything.

## If a structural check below fails
It raises `AssertionError` naming the failing check. A failed weight/statistic cross-check against
Gate 4's own recorded numbers means either the real Gold layer or an upstream artifact has drifted
since Gate 4 ran — re-run Gate 2/3/4 for real before trusting this gate's own output. This gate
never relaxes a tolerance, never reconstructs a barred column as a scoring input, and never
substitutes a fabricated number for a real one that could not be computed.

In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP7 Gate 5 decision layer / full-population reporting
notebook. Single consolidated code cell (platform convention). Idempotent - safe to re-run.
"""

import os
import sys
import json
import warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)

    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import (  # noqa: E402
    assert_within_ram_ceiling,
    configure_performance,
    load_resource_limits,
    memory_headroom_gb,
)

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)
_ram_ceiling_fraction = RESOURCE_LIMITS["ceilings"]["max_ram_fraction"]
print(f"[WARP] Headroom before heavy work: {memory_headroom_gb(_ram_ceiling_fraction)} GB")

# ============================================================
# SECTION 3: Heavy imports (only after WARP configuration)
# ============================================================
import pandas as pd  # noqa: E402
import polars as pl  # noqa: E402
import yaml  # noqa: E402

from utils.bp1_config_sync import write_gate_block  # noqa: E402
from features.bp7_decision_engine_features import (  # noqa: E402
    CANDIDATE_NAMES,
    DEFAULT_INTERVENTION_THRESHOLD,
    FINAL_OUTPUT_COLUMNS,
    load_bp2_friction_ordinal_ranks,
    load_upstream_validated_metrics,
    compute_bp4_join_coverage,
    compute_bp2_bp3_correlation,
    attach_normalized_signal_columns,
    compute_candidate_raw_weights,
    normalize_candidate_weights,
    score_priority_rule,
    benchmark_candidate,
    coverage_report,
    compute_priority_score_contribution_decomposition,
    summarize_contribution_decomposition,
    load_bp3_gold_tags_group,
    attach_tags_group_for_audit,
    compute_disparate_impact_audit,
    build_full_population_decision_records,
    summarize_recommended_action_breakdown,
    summarize_bp4_tier_intervention_crosstab,
)

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
ARTIFACTS_DIR = NOTEBOOKS_DIR / "bp7_customer_navigator_decision_engine" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

BP7_CONFIG_PATH = CONFIGS_DIR / "bp7_customer_navigator_decision_engine.yaml"
GOLD_PATH = DATA_PROCESSED / "cfpb_decision_engine_context_gold.parquet"
GATE4_SUMMARY_PATH = ARTIFACTS_DIR / "gate4_statistical_validation_explainability_summary.json"

BP3_ARTIFACTS_DIR = NOTEBOOKS_DIR / "bp3_complaint_escalation_prediction" / "artifacts"
BP3_GOLD_PATH = DATA_PROCESSED / "cfpb_intervention_escalation_gold.parquet"

RECORDS_CSV_PATH = ARTIFACTS_DIR / "gate5_full_population_decision_records.csv"
ACTION_BREAKDOWN_CSV_PATH = ARTIFACTS_DIR / "gate5_recommended_action_breakdown.csv"
TIER_CROSSTAB_CSV_PATH = ARTIFACTS_DIR / "gate5_bp4_tier_intervention_crosstab.csv"
DISPARATE_IMPACT_BREAKDOWN_CSV_PATH = ARTIFACTS_DIR / "gate5_disparate_impact_breakdown.csv"
SUMMARY_JSON_PATH = ARTIFACTS_DIR / "gate5_decision_layer_summary.json"

for p in (BP7_CONFIG_PATH, GOLD_PATH, GATE4_SUMMARY_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"Required input not found: {p}. Confirm BP7 Gate 4 has been real-run at least once."
        )

# ============================================================
# SECTION 4: Gate 4 prerequisite check (live) - never trusted from memory, re-read every run.
# ============================================================
with open(BP7_CONFIG_PATH, "r", encoding="utf-8") as f:
    full_config_text = f.read()
full_config = yaml.safe_load(full_config_text)
gate4_block_marker = (
    "# --- Gate 4 (Statistical Validation & Explainability) results " "(appended, idempotent overwrite) ---"
)
gate4_block_present = gate4_block_marker in full_config_text
gate4_confirmed = (
    gate4_block_present
    and full_config.get("gate4_champion_reproduced_bit_exact") is True
    and full_config.get("gate4_leakage_reconfirmed_clean") is True
    and full_config.get("gate4_disparate_impact_check_performed") is True
)
assert gate4_confirmed, (
    "BP7 Gate 4 does not appear to have completed successfully (gate4_block_present="
    f"{gate4_block_present}, gate4_champion_reproduced_bit_exact="
    f"{full_config.get('gate4_champion_reproduced_bit_exact')!r}, "
    f"gate4_leakage_reconfirmed_clean={full_config.get('gate4_leakage_reconfirmed_clean')!r}). "
    "Run Gate 4 for real before Gate 5."
)
RANDOM_STATE = full_config.get("random_state", 42)
RECORDED_CHAMPION = full_config["champion_rule_scheme"]
assert RECORDED_CHAMPION in CANDIDATE_NAMES, (
    f"[CHECK FAILED] Recorded champion_rule_scheme {RECORDED_CHAMPION!r} is not one of the known "
    f"CANDIDATE_NAMES {CANDIDATE_NAMES!r} - configs/bp7_customer_navigator_decision_engine.yaml "
    "has drifted from this module's own CANDIDATE_NAMES constant."
)
RECORDED_THRESHOLD = float(full_config.get("intervention_threshold", DEFAULT_INTERVENTION_THRESHOLD))
print(
    f"[OK] Gate 4 prerequisite confirmed - recorded champion_rule_scheme={RECORDED_CHAMPION!r}, "
    f"intervention_threshold={RECORDED_THRESHOLD}."
)

with open(GATE4_SUMMARY_PATH, "r", encoding="utf-8") as f:
    gate4_summary = json.load(f)

# ============================================================
# SECTION 5: Load the real Gold layer live, confirm row count still matches config's own recorded
# count (never assumed unchanged since Gate 4 ran).
# ============================================================
gold_pl = pl.read_parquet(GOLD_PATH)
live_row_count = gold_pl.height
row_count_matches_config = live_row_count == full_config.get("gold_layer_rows_written")
print(
    f"[OK] Real Gold layer loaded live: {live_row_count:,} rows x {gold_pl.width} cols "
    f"(matches config's own recorded gold_layer_rows_written: {row_count_matches_config})."
)
assert row_count_matches_config, (
    "Real Gold layer row count no longer matches the config's own recorded "
    f"gold_layer_rows_written ({full_config.get('gold_layer_rows_written')}) - re-run Gate "
    "2/3/4 before trusting Gate 5's own output below."
)

# ============================================================
# SECTION 6: Independently re-derive the FULL-PRECISION normalized weights for the recorded
# champion (never reading the rounded config values for scoring - see this notebook's own
# markdown cell for why). Cross-checked against config's recorded (rounded) values with the same
# tolerances Gate 4 already established for this identical rounding gap.
# ============================================================
print("\n" + "=" * 70)
print("SECTION 6: RE-DERIVING FULL-PRECISION CHAMPION WEIGHTS (never from rounded config)")
print("=" * 70)

correlation_result = compute_bp2_bp3_correlation(gold_pl)
reproduced_cramers_v = correlation_result["chi_square_cramers_v"]["cramers_v"]

upstream_metrics = load_upstream_validated_metrics(PROJECT_ROOT)
reproduced_bp4_coverage = compute_bp4_join_coverage(gold_pl)

raw_weights_by_candidate = compute_candidate_raw_weights(
    upstream_metrics, reproduced_bp4_coverage, reproduced_cramers_v
)
CHAMPION_WEIGHTS_NORMALIZED = normalize_candidate_weights(raw_weights_by_candidate[RECORDED_CHAMPION])
print(f"[OK] Re-derived full-precision normalized weights for {RECORDED_CHAMPION!r}:")
print(f"     {CHAMPION_WEIGHTS_NORMALIZED}")


def _close(a, b, tol: float) -> bool:
    if a is None or b is None:
        return a == b
    return abs(float(a) - float(b)) < tol


cramers_v_matches = _close(reproduced_cramers_v, full_config.get("bp2_bp3_cramers_v"), tol=1e-4)
bp4_coverage_matches = _close(reproduced_bp4_coverage, full_config.get("bp4_join_coverage"), tol=1e-4)
weights_match = (
    _close(CHAMPION_WEIGHTS_NORMALIZED["bp2"], full_config.get("champion_weight_bp2"), tol=1e-6)
    and _close(CHAMPION_WEIGHTS_NORMALIZED["bp3"], full_config.get("champion_weight_bp3"), tol=1e-6)
    and _close(CHAMPION_WEIGHTS_NORMALIZED["bp4"], full_config.get("champion_weight_bp4"), tol=1e-6)
)
print(
    f"[CHECK] cramers_v matches config (tol 1e-4): {cramers_v_matches} "
    f"(reproduced={reproduced_cramers_v:.6f}, config={full_config.get('bp2_bp3_cramers_v')})"
)
print(
    f"[CHECK] bp4_join_coverage matches config (tol 1e-4): {bp4_coverage_matches} "
    f"(reproduced={reproduced_bp4_coverage:.6f}, config={full_config.get('bp4_join_coverage')})"
)
print(f"[CHECK] champion weights match config's rounded values (tol 1e-6): {weights_match}")
assert cramers_v_matches and bp4_coverage_matches and weights_match, (
    "[CHECK FAILED] Freshly re-derived champion weights do not match "
    "configs/bp7_customer_navigator_decision_engine.yaml's own recorded Gate 3/4 values within "
    "tolerance - an upstream artifact has drifted since Gate 3/4 ran. Re-run Gate 2/3/4 for real "
    "before trusting Gate 5's own scoring below."
)

# ============================================================
# SECTION 7: Score the REAL FULL POPULATION with the champion's freshly re-derived, full-precision
# weights - Gate 1's own four named output fields (priority_score / intervention_flag /
# recommended_action / reason_codes), for every one of the real 1,048,575 CFPB rows.
# ============================================================
print("\n" + "=" * 70)
print("SECTION 7: FULL-POPULATION CHAMPION SCORING")
print("=" * 70)

friction_ordinal_ranks = load_bp2_friction_ordinal_ranks(PROJECT_ROOT)
signal_lazy = attach_normalized_signal_columns(gold_pl.lazy(), friction_ordinal_ranks)
scored_pl = score_priority_rule(signal_lazy, CHAMPION_WEIGHTS_NORMALIZED, RECORDED_THRESHOLD).collect()
assert scored_pl.height == live_row_count, (
    f"[CHECK FAILED] Scored row count ({scored_pl.height:,}) does not match the real full "
    f"population ({live_row_count:,})."
)

champion_stats = benchmark_candidate(
    scored_pl, RECORDED_CHAMPION, CHAMPION_WEIGHTS_NORMALIZED, raw_weights_by_candidate[RECORDED_CHAMPION]
)
print(
    f"[RESULT] coverage_pct={champion_stats['coverage_pct']}, "
    f"intervention_flag_rate={champion_stats['intervention_flag_rate']}, "
    f"bp3_agreement_rate={champion_stats['bp3_agreement_rate']}, "
    f"reason_codes_all_nonempty={champion_stats['reason_codes_all_nonempty']}"
)

coverage_pct_matches_gate3 = _close(
    champion_stats["coverage_pct"], full_config.get("champion_coverage_pct"), 1e-3
)
intervention_rate_matches_gate4 = _close(
    champion_stats["intervention_flag_rate"],
    full_config.get("gate4_reproduced_intervention_flag_rate"),
    tol=1e-6,
)
bp3_agreement_matches_gate4 = _close(
    champion_stats["bp3_agreement_rate"],
    full_config.get("gate4_bp3_agreement_rate_point_estimate"),
    tol=1e-6,
)
print(f"[CHECK] coverage_pct matches Gate 3's recorded value (tol 1e-3): {coverage_pct_matches_gate3}")
print(
    "[CHECK] intervention_flag_rate matches Gate 4's recorded point estimate (tol 1e-6): "
    f"{intervention_rate_matches_gate4}"
)
print(
    f"[CHECK] bp3_agreement_rate matches Gate 4's recorded point estimate (tol 1e-6): "
    f"{bp3_agreement_matches_gate4}"
)
assert coverage_pct_matches_gate3 and intervention_rate_matches_gate4 and bp3_agreement_matches_gate4, (
    "[CHECK FAILED] This gate's freshly-scored full population does not reproduce Gate 3/4's own "
    "recorded champion statistics within tolerance - an upstream artifact has drifted. Re-run "
    "Gate 2/3/4 for real before trusting this gate's own deliverable."
)
assert champion_stats["reason_codes_all_nonempty"], (
    "[CHECK FAILED] Not every scored row carries a non-empty reason_codes value - violates Gate "
    "1's own Section 5.1/7 requirement that every output row record which upstream fields drove it."
)

# ============================================================
# SECTION 8: Exact per-row contribution decomposition (reused unmodified from Gate 4).
# ============================================================
print("\n" + "=" * 70)
print("SECTION 8: EXACT CONTRIBUTION DECOMPOSITION")
print("=" * 70)

decomp_pl = compute_priority_score_contribution_decomposition(scored_pl, CHAMPION_WEIGHTS_NORMALIZED)
contribution_summary = summarize_contribution_decomposition(decomp_pl)
print(
    f"[CHECK] max_abs_reconstruction_error={contribution_summary['max_abs_reconstruction_error']} "
    f"(exact within tolerance: {contribution_summary['reconstruction_exact_within_tolerance']})"
)
assert contribution_summary["reconstruction_exact_within_tolerance"], (
    "[CHECK FAILED] contribution_bp2 + contribution_bp3 + contribution_bp4 does not reconstruct "
    "priority_score exactly for every scored row."
)

# ============================================================
# SECTION 9: Audit-only tags_group join + fresh disparate-impact re-derivation on THIS gate's own
# scored full population (reused unmodified from Gate 4), cross-checked against Gate 4's own
# recorded finding. tags_group never enters scoring - joined strictly after Section 7 above.
# ============================================================
print("\n" + "=" * 70)
print("SECTION 9: AUDIT-ONLY DISPARATE-IMPACT RE-DERIVATION (tags_group, ECOA/Reg B)")
print("=" * 70)

tags_group_pl = None
if BP3_GOLD_PATH.exists():
    tags_group_pl = load_bp3_gold_tags_group(BP3_GOLD_PATH)
    audited_pl = attach_tags_group_for_audit(scored_pl, tags_group_pl)
    disparate_impact_result = compute_disparate_impact_audit(audited_pl)
    print(
        f"[RESULT] join_coverage_pct={disparate_impact_result['join_coverage_pct']}, "
        f"adverse_impact_ratio={disparate_impact_result['adverse_impact_ratio']}, "
        f"flagged_four_fifths_rule={disparate_impact_result['flagged_four_fifths_rule']}"
    )
    adverse_impact_matches_gate4 = _close(
        disparate_impact_result["adverse_impact_ratio"],
        full_config.get("gate4_disparate_impact_adverse_impact_ratio"),
        tol=1e-6,
    )
    print(
        f"[CHECK] adverse_impact_ratio matches Gate 4's recorded value (tol 1e-6): "
        f"{adverse_impact_matches_gate4}"
    )
    assert adverse_impact_matches_gate4, (
        "[CHECK FAILED] This gate's freshly re-derived disparate-impact ratio does not match "
        "Gate 4's own recorded value within tolerance - re-run Gate 4 for real before trusting "
        "this gate's own output."
    )
    pd.DataFrame(disparate_impact_result["group_breakdown"]).to_csv(
        DISPARATE_IMPACT_BREAKDOWN_CSV_PATH, index=False
    )
    print(f"[SAVED] {DISPARATE_IMPACT_BREAKDOWN_CSV_PATH.relative_to(PROJECT_ROOT)}")
else:
    disparate_impact_result = None
    adverse_impact_matches_gate4 = None
    print(
        f"[LIMITATION] {BP3_GOLD_PATH} not found - BP3's own real Gold layer is unavailable. "
        "tags_group will be NOT_AVAILABLE_BUNDLE_NOT_YET_PERSISTED for every row in the final "
        "output, and no fresh disparate-impact breakdown is written this run (Gate 4's own "
        "already-real-run-confirmed finding, carried forward in this gate's summary JSON below, "
        "remains the authoritative one)."
    )

# ============================================================
# SECTION 10: Assemble and write the real, final full-population decision-record deliverable -
# one row per real complaint, Gate 1's own four named output fields plus full context.
# ============================================================
print("\n" + "=" * 70)
print("SECTION 10: WRITING THE FULL-POPULATION DECISION RECORDS")
print("=" * 70)

records_pl = build_full_population_decision_records(scored_pl, decomp_pl, tags_group_pl)
assert records_pl.height == live_row_count, (
    f"[CHECK FAILED] Final decision-record count ({records_pl.height:,}) does not match the real "
    f"full population ({live_row_count:,})."
)
assert (
    list(records_pl.columns) == FINAL_OUTPUT_COLUMNS
), "[CHECK FAILED] Final decision-record columns do not exactly match FINAL_OUTPUT_COLUMNS."
records_pl.write_csv(RECORDS_CSV_PATH)
print(
    f"[SAVED] {RECORDS_CSV_PATH.relative_to(PROJECT_ROOT)} "
    f"({records_pl.height:,} real per-complaint decision records, one row per real complaint)"
)

# ============================================================
# SECTION 11: Real reporting rollups - recommended_action breakdown, BP4-tier x intervention_flag
# cross-tab, and the real per-field upstream-coverage report (Gate 2's own coverage_report(),
# reused unmodified).
# ============================================================
print("\n" + "=" * 70)
print("SECTION 11: REPORTING ROLLUPS")
print("=" * 70)

action_breakdown_pl = summarize_recommended_action_breakdown(records_pl)
action_breakdown_pl.write_csv(ACTION_BREAKDOWN_CSV_PATH)
print(f"[SAVED] {ACTION_BREAKDOWN_CSV_PATH.relative_to(PROJECT_ROOT)}")
print(action_breakdown_pl.to_pandas().to_string(index=False))

tier_crosstab_pl = summarize_bp4_tier_intervention_crosstab(records_pl)
tier_crosstab_pl.write_csv(TIER_CROSSTAB_CSV_PATH)
print(f"\n[SAVED] {TIER_CROSSTAB_CSV_PATH.relative_to(PROJECT_ROOT)}")
print(tier_crosstab_pl.to_pandas().to_string(index=False))

upstream_coverage = coverage_report(gold_pl)
print(
    f"\n[RESULT] Real upstream-field coverage (Gate 2's own coverage_report(), reused): {upstream_coverage}"
)

action_breakdown_records = action_breakdown_pl.to_dicts()
tier_crosstab_records = tier_crosstab_pl.to_dicts()
n_action_rows_check = sum(r["n_rows"] for r in action_breakdown_records)
n_tier_rows_check = sum(r["n_rows"] for r in tier_crosstab_records)

# ============================================================
# SECTION 12: Compliance touchpoints (Master Plan Section 8's Gate 5 row) - stated honestly,
# never silently skipped. Unlike every upstream BP's own Gate 5, "Decision" is NOT Not-Applicable
# here - this IS the cross-BP decision engine those BPs deferred to.
# ============================================================
compliance_touchpoint = {
    "udaap_language_review": (
        "Not Applicable to BP7 Gate 5 - recommended_action is Gate 1's own named deterministic, "
        "reason-code-keyed lookup, never GenAI-generated text. Real GenAI-drafted customer-facing "
        "text (subject to UDAAP review) is scoped to BP6 per the Master Plan, the identical "
        "standing scope decision confirmed at every upstream BP's own Gate 5."
    ),
    "nist_ai_rmf_measure_manage": (
        "Not Applicable to BP7 Gate 5 for the same reason - no GenAI output is produced here. "
        "Applies at BP6."
    ),
    "ecoa_reg_b_disparate_impact_monitoring": (
        "Applicable - Section 9 above freshly re-derives the disparate-impact audit on this "
        "gate's own scored full population and cross-checks it against Gate 4's own already-real"
        "-run-confirmed finding, rather than simply re-printing Gate 4's number. Monitoring "
        "signal for a human reviewer, not a legal determination of ECOA/Reg B compliance - the "
        "identical limitation Gate 4 and BP3's own Gate 4 both stated, not softened here."
    ),
    "bp7_is_the_cross_bp_decision_engine": (
        "Unlike every upstream BP's own Gate 5 (which each stated the cross-BP priority + "
        "intervention-risk decision was Not Applicable at their own level and named it as BP7's "
        "job), this gate IS that decision engine - priority_score/intervention_flag/"
        "recommended_action/reason_codes are computed here, for real, for the full real "
        "population, using Gate 3's already-selected, Gate 4-reproduced champion rule scheme."
    ),
    "genai_api_used": False,
}
print("\n=== COMPLIANCE TOUCHPOINTS ===")
for key, value in compliance_touchpoint.items():
    print(f"  {key}: {value}")

# ============================================================
# SECTION 13: Write the Gate 5 master summary JSON artifact.
# ============================================================
gate5_summary = {
    "bp_id": "bp7",
    "gate": 5,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "live_row_count": live_row_count,
    "champion_rule_scheme": RECORDED_CHAMPION,
    "champion_weights_normalized": CHAMPION_WEIGHTS_NORMALIZED,
    "intervention_threshold": RECORDED_THRESHOLD,
    "weight_rederivation_cross_check": {
        "cramers_v_matches_config": cramers_v_matches,
        "bp4_coverage_matches_config": bp4_coverage_matches,
        "weights_match_config": weights_match,
    },
    "champion_stats": {
        "coverage_pct": champion_stats["coverage_pct"],
        "intervention_flag_rate": champion_stats["intervention_flag_rate"],
        "bp3_agreement_rate": champion_stats["bp3_agreement_rate"],
        "avg_reason_codes_per_row": champion_stats["avg_reason_codes_per_row"],
        "reason_codes_all_nonempty": champion_stats["reason_codes_all_nonempty"],
    },
    "cross_checks_vs_gate3_gate4": {
        "coverage_pct_matches_gate3": coverage_pct_matches_gate3,
        "intervention_flag_rate_matches_gate4": intervention_rate_matches_gate4,
        "bp3_agreement_rate_matches_gate4": bp3_agreement_matches_gate4,
        "adverse_impact_ratio_matches_gate4": adverse_impact_matches_gate4,
    },
    "gate4_bootstrap_ci_carried_forward": {
        "intervention_flag_rate_ci_95": full_config.get("gate4_intervention_flag_rate_bootstrap_ci_95"),
        "bp3_agreement_rate_ci_95": full_config.get("gate4_bp3_agreement_rate_bootstrap_ci_95"),
        "n_bootstrap": full_config.get("gate4_bootstrap_n_resamples"),
    },
    "contribution_decomposition_summary": contribution_summary,
    "disparate_impact_audit": (
        {
            "join_coverage_pct": disparate_impact_result["join_coverage_pct"],
            "adverse_impact_ratio": disparate_impact_result["adverse_impact_ratio"],
            "flagged_four_fifths_rule": disparate_impact_result["flagged_four_fifths_rule"],
            "lowest_selection_rate_group": disparate_impact_result["lowest_selection_rate_group"],
            "highest_selection_rate_group": disparate_impact_result["highest_selection_rate_group"],
            "breakdown_path": str(DISPARATE_IMPACT_BREAKDOWN_CSV_PATH.relative_to(PROJECT_ROOT).as_posix()),
        }
        if disparate_impact_result is not None
        else {
            "performed_this_run": False,
            "reason": f"{BP3_GOLD_PATH} not found this run.",
            "gate4_recorded_adverse_impact_ratio": full_config.get(
                "gate4_disparate_impact_adverse_impact_ratio"
            ),
        }
    ),
    "upstream_field_coverage": upstream_coverage,
    "recommended_action_breakdown": action_breakdown_records,
    "bp4_tier_intervention_crosstab": tier_crosstab_records,
    "n_rows_covered_by_action_breakdown": n_action_rows_check,
    "n_rows_covered_by_tier_crosstab": n_tier_rows_check,
    "compliance_touchpoint": compliance_touchpoint,
    "records_csv_path": str(RECORDS_CSV_PATH.relative_to(PROJECT_ROOT).as_posix()),
    "action_breakdown_csv_path": str(ACTION_BREAKDOWN_CSV_PATH.relative_to(PROJECT_ROOT).as_posix()),
    "tier_crosstab_csv_path": str(TIER_CROSSTAB_CSV_PATH.relative_to(PROJECT_ROOT).as_posix()),
}
with open(SUMMARY_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(gate5_summary, f, indent=2, default=str)
print(f"\n[SAVED] {SUMMARY_JSON_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 14: Write the Gate 5 config block (marker-based, order-independent - reuses
# bp1_config_sync.py unmodified). `status` is deliberately NOT touched - this project's own
# established BP7 convention (Gate 2, Gate 3, and Gate 4 all left it untouched).
# ============================================================
gate5_marker = "# --- Gate 5 (Decision Layer & Reporting) results (appended, idempotent overwrite) ---"
FINAL_ADVERSE_IMPACT_RATIO = (
    disparate_impact_result["adverse_impact_ratio"]
    if disparate_impact_result is not None
    else full_config.get("gate4_disparate_impact_adverse_impact_ratio")
)
ALL_CROSS_CHECKS_PASSED = bool(
    coverage_pct_matches_gate3
    and intervention_rate_matches_gate4
    and bp3_agreement_matches_gate4
    and (adverse_impact_matches_gate4 in (True, None))
)
gate5_block_lines = [
    f'gate5_champion_rule_scheme: "{RECORDED_CHAMPION}"',
    f"gate5_intervention_threshold: {RECORDED_THRESHOLD}",
    f"gate5_n_decision_records: {records_pl.height}",
    f"gate5_coverage_pct: {champion_stats['coverage_pct']}",
    f"gate5_intervention_flag_rate: {champion_stats['intervention_flag_rate']}",
    f"gate5_bp3_agreement_rate: {champion_stats['bp3_agreement_rate']}",
    "gate5_contribution_reconstruction_exact: "
    f"{contribution_summary['reconstruction_exact_within_tolerance']}",
    f"gate5_disparate_impact_rederived_this_run: {disparate_impact_result is not None}",
    f"gate5_disparate_impact_adverse_impact_ratio: {FINAL_ADVERSE_IMPACT_RATIO}",
    f"gate5_weight_rederivation_matches_config: {weights_match}",
    f"gate5_cross_checks_vs_gate3_gate4_all_passed: {ALL_CROSS_CHECKS_PASSED}",
    "genai_api_used: false",
    f'records_csv_path: "{RECORDS_CSV_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f'action_breakdown_csv_path: "{ACTION_BREAKDOWN_CSV_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f'tier_crosstab_csv_path: "{TIER_CROSSTAB_CSV_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f'gate5_summary_path: "{SUMMARY_JSON_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f'gate5_generated_at_utc: "{gate5_summary["generated_at_utc"]}"',
]
write_gate_block(BP7_CONFIG_PATH, gate5_marker, gate5_block_lines)
print(f"[SAVED] gate5 block written to {BP7_CONFIG_PATH.relative_to(PROJECT_ROOT)} (status untouched)")

with open(BP7_CONFIG_PATH, "r", encoding="utf-8") as f:
    _post_write_config_text = f.read()
gate5_block_actually_written = gate5_marker in _post_write_config_text
_post_write_config = yaml.safe_load(_post_write_config_text)
status_untouched = _post_write_config.get("status") == full_config.get("status")

# ============================================================
# SECTION 15: Structural integrity checks - raise AssertionError, never silently pass.
# ============================================================
checks = {
    "gate4_prerequisite_confirmed": gate4_confirmed,
    "gold_layer_row_count_matches_config": row_count_matches_config,
    "champion_weight_rederivation_matches_config": weights_match,
    "cramers_v_rederivation_matches_config": cramers_v_matches,
    "bp4_coverage_rederivation_matches_config": bp4_coverage_matches,
    "scored_row_count_matches_population": scored_pl.height == live_row_count,
    "coverage_pct_matches_gate3_recorded": coverage_pct_matches_gate3,
    "intervention_flag_rate_matches_gate4_recorded": intervention_rate_matches_gate4,
    "bp3_agreement_rate_matches_gate4_recorded": bp3_agreement_matches_gate4,
    "reason_codes_all_nonempty": champion_stats["reason_codes_all_nonempty"],
    "contribution_decomposition_reconstruction_exact": contribution_summary[
        "reconstruction_exact_within_tolerance"
    ],
    "disparate_impact_matches_gate4_recorded_or_honestly_skipped": adverse_impact_matches_gate4
    in (True, None),
    "final_records_row_count_matches_population": records_pl.height == live_row_count,
    "final_records_columns_match_final_output_columns": list(records_pl.columns) == FINAL_OUTPUT_COLUMNS,
    "action_breakdown_covers_full_population": n_action_rows_check == live_row_count,
    "tier_crosstab_covers_full_population": n_tier_rows_check == live_row_count,
    "compliance_touchpoint_documented": bool(compliance_touchpoint),
    "records_csv_written": RECORDS_CSV_PATH.exists(),
    "action_breakdown_csv_written": ACTION_BREAKDOWN_CSV_PATH.exists(),
    "tier_crosstab_csv_written": TIER_CROSSTAB_CSV_PATH.exists(),
    "summary_json_written": SUMMARY_JSON_PATH.exists(),
    "config_gate5_block_written": gate5_block_actually_written,
    "status_field_untouched": status_untouched,
}

print("\n=== INTEGRITY CHECKS ===")
for check_name, passed in checks.items():
    result_label = "[PASS]" if passed else "[FAIL]"
    print(f"{result_label} {check_name}")
    assert passed, f"[CHECK FAILED] {check_name}"

print(
    f"\n[ALL CHECKS PASSED] BP7 Gate 5 complete - {records_pl.height:,} real full-population "
    f"decision records written (champion='{RECORDED_CHAMPION}', "
    f"intervention_flag_rate={champion_stats['intervention_flag_rate']}, "
    f"bp3_agreement_rate={champion_stats['bp3_agreement_rate']}). Disparate-impact/ECOA Reg B: "
    f"adverse_impact_ratio={FINAL_ADVERSE_IMPACT_RATIO} "
    "(cross-checked against Gate 4's own recorded finding). No GenAI API used. status left "
    "untouched (BP7's own established convention). Proceed to BP7 Gate 6 (Productization, "
    "Monitoring & Governance) next."
)
